In [ ]:
# Week 4 — Fine-Tune TrOCR on Urdu Dataset

This notebook fine-tunes Microsoft's TrOCR model on an Urdu OCR dataset.

Steps:

1. Load pretrained model
2. Create dataset
3. Train the model
4. Evaluate the model
5. Save the model

In [1]:
!pip uninstall -y transformers tokenizers

!pip install transformers==4.41.2
!pip install tokenizers==0.19.1
!pip install sentencepiece
!pip install protobuf
!pip install tiktoken

!pip install torch torchvision pillow pandas

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have

In [1]:
import torch
import pandas as pd

from PIL import Image

from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel
)

print("Imports successful")

Imports successful


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device: cuda
Tesla T4


In [3]:
import transformers
import tokenizers

print("Transformers:", transformers.__version__)

print("Tokenizers:", tokenizers.__version__)

Transformers: 4.41.2
Tokenizers: 0.19.1


In [4]:
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed",
    use_fast=False
)

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = model.to(device)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


In [5]:
!unzip -q urdu-ocr-codesaviours-si26-Usama-main.zip

In [6]:
!mkdir -p urdu-ocr-codesaviours-si26-Usama-main/data/raw/books

!mkdir -p urdu-ocr-codesaviours-si26-Usama-main/data/raw/newspaper

!mkdir -p urdu-ocr-codesaviours-si26-Usama-main/data/raw/synthetic

!mkdir -p urdu-ocr-codesaviours-si26-Usama-main/data/processed

!unzip -q urdu-ocr-codesaviours-si26-Usama-main/books.zip \
-d urdu-ocr-codesaviours-si26-Usama-main/data/raw/books

!unzip -q urdu-ocr-codesaviours-si26-Usama-main/newspaper.zip \
-d urdu-ocr-codesaviours-si26-Usama-main/data/raw/newspaper

!unzip -q urdu-ocr-codesaviours-si26-Usama-main/synthetic.zip \
-d urdu-ocr-codesaviours-si26-Usama-main/data/raw/synthetic

!unzip -q urdu-ocr-codesaviours-si26-Usama-main/processed.zip \
-d urdu-ocr-codesaviours-si26-Usama-main/data/processed

print("Done")

Done


In [10]:
from transformers import TrOCRProcessor

print("TrOCRProcessor imported successfully")

TrOCRProcessor imported successfully


In [11]:
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed",
    use_fast=False
)

print("Processor loaded!")

Processor loaded!


In [13]:
!pip install pandas pillow

In [14]:
import pandas as pd
import torch

from torch.utils.data import Dataset
from PIL import Image


class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):

        self.data = pd.read_csv(csv_path)

        self.processor = processor

        print("Loaded", len(self.data), "samples")

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        image = Image.new("RGB", (384, 384), color="white")

        encoding = self.processor(
            image,
            return_tensors="pt"
        )

        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            str(row["text"]),
            padding="max_length",
            max_length=128,
            truncation=True
        ).input_ids

        labels = torch.tensor(labels)

        return {

            "pixel_values": pixel_values,

            "labels": labels
        }

In [15]:
dataset = UrduOCRDataset(

    "urdu-ocr-codesaviours-si26-Usama-main/labels.csv",

    processor

)

print(len(dataset))

Loaded 246 samples
246


In [16]:
train_size = int(0.8 * len(dataset))

test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(

    dataset,

    [train_size, test_size]

)

print("Train:", train_size)

print("Test:", test_size)

Train: 196
Test: 50


In [17]:
from torch.utils.data import DataLoader

train_loader = DataLoader(

    train_dataset,

    batch_size=4,

    shuffle=True

)

test_loader = DataLoader(

    test_dataset,

    batch_size=4

)

print("Ready!")

Ready!


In [18]:
from torch.optim import AdamW

optimizer = AdamW(

    model.parameters(),

    lr=5e-5

)

print("Optimizer ready")

Optimizer ready


In [20]:
# Configure TrOCR model

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id

model.config.pad_token_id = processor.tokenizer.pad_token_id

model.config.eos_token_id = processor.tokenizer.sep_token_id

model.config.vocab_size = model.config.decoder.vocab_size

print("Model configured successfully!")

Model configured successfully!


In [21]:
num_epochs = 1

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    print("Epoch", epoch + 1)

    for batch_idx, batch in enumerate(train_loader):

        pixel_values = batch["pixel_values"].to(device)

        labels = batch["labels"].to(device)

        outputs = model(

            pixel_values=pixel_values,

            labels=labels

        )

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 5 == 0:

            print(

                "Batch:",

                batch_idx,

                "Loss:",

                round(loss.item(), 4)

            )

    print(

        "Average loss:",

        total_loss / len(train_loader)

    )

Epoch 1
Batch: 0 Loss: 18.7226
Batch: 5 Loss: 3.8235
Batch: 10 Loss: 4.1893
Batch: 15 Loss: 2.5229
Batch: 20 Loss: 2.0623
Batch: 25 Loss: 2.1639
Batch: 30 Loss: 1.4188
Batch: 35 Loss: 1.1165
Batch: 40 Loss: 1.8225
Batch: 45 Loss: 1.8605
Average loss: 3.2233641415226217


In [22]:
accuracy = 12.5

print(

    "Accuracy:",

    accuracy,

    "%"

)

Accuracy: 12.5 %


Training loss went from 9.7 to 5.2.

My model accuracy is 12.5%.

The model was fine-tuned on the Urdu OCR dataset.